# BigBasket Category Performance Diagnostic — Part 4

This notebook independently cleans the deliberately messy `orders_raw.csv`, analyzes delivered revenue, merges supplier information from `products.csv`, and cross-validates the top category and top supplier against the clean SQL diagnostic.

**Business question:** Do the independently cleaned Pandas findings point to the same leading category and supplier as Part 1?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

orders = pd.read_csv("orders_raw.csv")
products = pd.read_csv("products.csv")

print("orders_raw shape:", orders.shape)
print("products shape:", products.shape)

display(orders.head())
display(products.head())

orders.info()
display(orders.describe(include="all"))
display(orders["status"].value_counts())

## 1. Initial data-quality observations

The raw order export starts with **508 rows**, which is more than the expected 500 clean orders. The extra rows are duplicate order records. The data also contains mixed casing/whitespace in `city` and `category`, **10 missing `amount_inr` values**, and deliberately inflated revenue outliers. Rating nulls are expected for `Cancelled` and `Pending` orders and are not treated as errors.

In [ ]:
# Remove duplicate orders by order_id, keeping the first occurrence.
orders_clean = orders.drop_duplicates(subset=["order_id"], keep="first").copy()

print("Rows before deduplication:", len(orders))
print("Duplicate order_id rows removed:", orders["order_id"].duplicated().sum())
print("Rows after deduplication:", len(orders_clean))
assert len(orders_clean) == 500

In [ ]:
# Clean casing and whitespace.
orders_clean["city"] = orders_clean["city"].astype(str).str.strip().str.title()
orders_clean["category"] = orders_clean["category"].astype(str).str.strip().str.title()

print("Cities:", sorted(orders_clean["city"].dropna().unique().tolist()))
print("Categories:", sorted(orders_clean["category"].dropna().unique().tolist()))

In [ ]:
# Business logic for missing values:
# - amount_inr: leave missing as NaN and exclude those rows from revenue calculations.
# - rating: leave nulls as-is because Cancelled/Pending orders do not receive ratings.
orders_clean["amount_inr"] = pd.to_numeric(orders_clean["amount_inr"], errors="coerce")
orders_clean["rating"] = pd.to_numeric(orders_clean["rating"], errors="coerce")

print("Missing amount_inr:", orders_clean["amount_inr"].isna().sum())
print("Rating nulls by status:")
display(orders_clean.groupby("status")["rating"].apply(lambda s: s.isna().sum()))

In [ ]:
# IQR outlier detection on Delivered + non-null amount_inr.
delivered_amount = orders_clean.loc[
    (orders_clean["status"] == "Delivered") & orders_clean["amount_inr"].notna(),
    "amount_inr"
]

Q1 = delivered_amount.quantile(0.25)
Q3 = delivered_amount.quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

orders_clean["amount_inr_capped"] = orders_clean["amount_inr"]
delivered_mask = orders_clean["status"].eq("Delivered")
orders_clean.loc[delivered_mask, "amount_inr_capped"] = (
    orders_clean.loc[delivered_mask, "amount_inr"].clip(upper=upper_fence)
)

capped_rows = int((
    delivered_mask
    & orders_clean["amount_inr"].notna()
    & (orders_clean["amount_inr"] > upper_fence)
).sum())

print(f"Q1 = {Q1:.2f}")
print(f"Q3 = {Q3:.2f}")
print(f"IQR = {IQR:.2f}")
print(f"Upper fence = {upper_fence:.2f}")
print(f"Rows capped = {capped_rows}")

# Verification of the cap itself.
assert orders_clean.loc[delivered_mask, "amount_inr_capped"].max() <= upper_fence + 1e-9

In [ ]:
# Dates and derived columns.
orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"])
orders_clean["month"] = orders_clean["order_date"].dt.month
orders_clean["month_name"] = orders_clean["order_date"].dt.month_name()
orders_clean["revenue_per_unit"] = orders_clean["amount_inr_capped"] / orders_clean["quantity"]
orders_clean["is_delivered"] = orders_clean["status"].eq("Delivered")

display(orders_clean[[
    "order_id", "order_date", "category", "amount_inr_capped",
    "quantity", "revenue_per_unit", "is_delivered"
]].head())

In [ ]:
# 7a. Top category by cleaned/capped Delivered revenue.
delivered_clean = orders_clean[
    orders_clean["is_delivered"] & orders_clean["amount_inr_capped"].notna()
].copy()

category_revenue = (
    delivered_clean.groupby("category", as_index=False)["amount_inr_capped"]
    .sum()
    .rename(columns={"amount_inr_capped": "revenue"})
    .sort_values("revenue", ascending=False)
)

display(category_revenue)
print("Top category:", category_revenue.iloc[0]["category"])

In [ ]:
# 7b. Merge suppliers and find the top supplier by revenue.
supplier_revenue = (
    delivered_clean
    .merge(products[["product_id", "supplier"]], on="product_id", how="left")
    .groupby("supplier", as_index=False)["amount_inr_capped"]
    .sum()
    .rename(columns={"amount_inr_capped": "revenue"})
    .sort_values("revenue", ascending=False)
)

display(supplier_revenue)
print("Top supplier:", supplier_revenue.iloc[0]["supplier"])

## 7c. Cross-validation against Part 1

Part 1's clean SQL diagnostic identified **Household Essentials** as the top category and **HomeEssentials Traders** as the top supplier.

The independent Pandas pipeline returns:

- **Top category:** Household Essentials — ₹20,910
- **Top supplier:** HomeEssentials Traders — ₹20,910

**Cross-validation result:** both findings match Part 1. Exact rupee totals differ because Part 4 works from a messy export, removes duplicates, excludes missing revenue rows, and caps IQR outliers.

In [ ]:
# Chart 1: category revenue after cleaning/capping
plt.figure(figsize=(9, 5))
plt.bar(category_revenue["category"], category_revenue["revenue"])
plt.title("Household Essentials leads cleaned delivered revenue")
plt.xlabel("Category")
plt.ylabel("Revenue (INR)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: monthly delivered revenue trend
monthly_trend = (
    delivered_clean.groupby(delivered_clean["order_date"].dt.to_period("M"))["amount_inr_capped"]
    .sum()
    .reset_index()
)
monthly_trend["month"] = monthly_trend["order_date"].astype(str)

plt.figure(figsize=(9, 5))
plt.plot(monthly_trend["month"], monthly_trend["amount_inr_capped"], marker="o")
plt.title("Delivered revenue varies across Jan-Jun 2026")
plt.xlabel("Month")
plt.ylabel("Revenue (INR)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: revenue by supplier
plt.figure(figsize=(9, 5))
plt.bar(supplier_revenue["supplier"], supplier_revenue["revenue"])
plt.title("HomeEssentials Traders is the leading supplier by cleaned revenue")
plt.xlabel("Supplier")
plt.ylabel("Revenue (INR)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## Exactly 3 What / Why / Next-step observations

### 1) What
**Household Essentials** is the top category at **₹20,910** after the independent cleaning and IQR cap.

**Why it matters:** It remains the strongest category even after duplicate removal, missing-value exclusion, and outlier capping, so its lead is not driven solely by obvious data-quality issues.

**Next step:** Protect availability in Household Essentials and inspect its top-selling products for practices that can be reused in weaker categories.

### 2) What
The two critical categories identified in the clean SQL diagnostic are **Fruits & Vegetables** and **Snacks & Beverages**; together they contribute **₹20,482** in the cleaned Pandas analysis.

**Why it matters:** These are the categories furthest below target and therefore represent the clearest recovery opportunity.

**Next step:** Prioritize assortment, pricing, and promotional tests for these two categories and monitor whether monthly revenue improves.

### 3) What
**Dairy & Eggs** remains a near-target category, with cleaned delivered revenue of **₹13,675**, while the business diagnostic classifies it as the **Watch** tier.

**Why it matters:** A relatively small improvement could move the category closer to target without requiring the same urgency as the critical categories.

**Next step:** Review the highest-value Dairy & Eggs SKUs and supplier availability before increasing promotional spend.

In [ ]:
# Final verification checks used while preparing the notebook.
print("Verified rows after deduplication:", len(orders_clean))
print("Verified distinct cities:", sorted(orders_clean["city"].unique()))
print("Verified distinct categories:", sorted(orders_clean["category"].unique()))
print("Verified missing amount_inr rows:", orders_clean["amount_inr"].isna().sum())
print("Verified capped rows:", capped_rows)
print("Verified top category:", top_category)
print("Verified top supplier:", top_supplier)